Movie Genre Classification

Importing Libraries

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
data=pd.read_csv("/content/description.txt")
print(data)

                                         Train data:
0             ID ::: TITLE ::: GENRE ::: DESCRIPTION
1             ID ::: TITLE ::: GENRE ::: DESCRIPTION
2             ID ::: TITLE ::: GENRE ::: DESCRIPTION
3             ID ::: TITLE ::: GENRE ::: DESCRIPTION
4                                         Test data:
5                       ID ::: TITLE ::: DESCRIPTION
6                       ID ::: TITLE ::: DESCRIPTION
7                       ID ::: TITLE ::: DESCRIPTION
8                       ID ::: TITLE ::: DESCRIPTION
9                                            Source:
10  ftp://ftp.fu-berlin.de/pub/misc/movies/database/


CREATE AND APPLY FUNCTION TO SEPARATE BY :::

In [2]:
def load_data(file_path):
  with open(file_path,'r',encoding='utf-8')as f:
    data=f.readlines()
  data=[line.strip().split(":::")for line in data]
  return data

In [3]:
train_data=load_data("/content/train_data.txt")
train_df=pd.DataFrame(train_data,columns=['ID','TITLE','GENRE','DESCRIPTION'])
test_data=load_data("/content/test_data.txt")
test_df=pd.DataFrame(test_data,columns=['ID','TITLE','DESCRIPTION'])
test_data_solution=load_data("/content/test_data_solution.txt")
test_solution_df=pd.DataFrame(test_data_solution,columns=['ID','TITLE','GENRE','DESCRIPTION'])

In [4]:
print("Train data: ")
train_df

Train data: 


,ID,TITLE,GENRE,DESCRIPTION
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous r...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...
...,...,...,...,...
54209,54210,"""Bonino"" (1953)",comedy,This short-lived NBC live sitcom centered on ...
54210,54211,Dead Girls Don't Cry (????),horror,The NEXT Generation of EXPLOITATION. The sist...
54211,54212,Ronald Goedemondt: Ze bestaan echt (2008),documentary,"Ze bestaan echt, is a stand-up comedy about g..."
54212,54213,Make Your Own Bed (1944),comedy,Walter and Vivian live in the country and hav...


In [5]:
print("\n Test data:")
test_df


 Test data:


,ID,TITLE,DESCRIPTION
0,1,Edgar's Lunch (1998),"L.R. Brane loves his life - his car, his apar..."
1,2,La guerra de papá (1977),"Spain, March 1964: Quico is a very naughty ch..."
2,3,Off the Beaten Track (2010),One year in the life of Albin and his family ...
3,4,Meu Amigo Hindu (2015),"His father has died, he hasn't spoken with hi..."
4,5,Er nu zhai (1955),Before he was known internationally as a mart...
...,...,...,...
54195,54196,"""Tales of Light & Dark"" (2013)","Covering multiple genres, Tales of Light & Da..."
54196,54197,Der letzte Mohikaner (1965),As Alice and Cora Munro attempt to find their...
54197,54198,Oliver Twink (2007),A movie 169 years in the making. Oliver Twist...
54198,54199,Slipstream (1973),"Popular, but mysterious rock D.J Mike Mallard..."


FEATURE EXTRACTION:TF-IDF

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=10000)
x_train_tfidf=vectorizer.fit_transform(train_df['DESCRIPTION'])
x_test_tfidf=vectorizer.transform(test_df['DESCRIPTION'])
print(f"training data shape:{x_train_tfidf.shape}")
print(f"testing data shape:{x_test_tfidf.shape}")


training data shape:(54214, 10000)
testing data shape:(54200, 10000)


ENCODING THE TARGET LABELS

In [7]:
from sklearn.preprocessing import LabelEncoder
label_encoder=LabelEncoder()
y_train=label_encoder.fit_transform(train_df['GENRE'])
print(f"unique genres in the data is{label_encoder.classes_}")

unique genres in the data is[' action ' ' adult ' ' adventure ' ' animation ' ' biography ' ' comedy '
 ' crime ' ' documentary ' ' drama ' ' family ' ' fantasy ' ' game-show '
 ' history ' ' horror ' ' music ' ' musical ' ' mystery ' ' news '
 ' reality-tv ' ' romance ' ' sci-fi ' ' short ' ' sport ' ' talk-show '
 ' thriller ' ' war ' ' western ']


MODEL-BUILDING LOGISTIC REGRESSION

In [8]:
from sklearn.linear_model import LogisticRegression
lr_model=LogisticRegression(max_iter=200)
lr_model.fit(x_train_tfidf,y_train)

y_predict=lr_model.predict(x_test_tfidf)
predicted_genres=label_encoder.inverse_transform(y_predict)
test_df['Predicted_Genre']=predicted_genres
test_df[['TITLE','Predicted_Genre']]

,TITLE,Predicted_Genre
0,Edgar's Lunch (1998),drama
1,La guerra de papá (1977),drama
2,Off the Beaten Track (2010),documentary
3,Meu Amigo Hindu (2015),drama
4,Er nu zhai (1955),drama
...,...,...
54195,"""Tales of Light & Dark"" (2013)",drama
54196,Der letzte Mohikaner (1965),drama
54197,Oliver Twink (2007),comedy
54198,Slipstream (1973),drama


In [9]:
test_df['Predicted_Genre']=predicted_genres
merged_df=pd.merge(test_solution_df[['ID','GENRE']],test_df[['ID','Predicted_Genre']],on='ID')

In [10]:
from sklearn.metrics import accuracy_score,classification_report


accuracy = accuracy_score(merged_df['GENRE'],merged_df['Predicted_Genre'])
print("Validation Accuracy:", accuracy)
print(classification_report(merged_df['GENRE'],merged_df['Predicted_Genre']))


Validation Accuracy: 0.5945387453874539
               precision    recall  f1-score   support

      action        0.51      0.30      0.37      1314
       adult        0.65      0.24      0.35       590
   adventure        0.67      0.16      0.25       775
   animation        0.61      0.04      0.08       498
   biography        0.00      0.00      0.00       264
      comedy        0.54      0.60      0.57      7446
       crime        0.41      0.03      0.06       505
 documentary        0.68      0.87      0.76     13096
       drama        0.55      0.78      0.65     13612
      family        0.49      0.08      0.14       783
     fantasy        0.61      0.03      0.06       322
   game-show        0.90      0.49      0.64       193
     history        0.00      0.00      0.00       243
      horror        0.66      0.57      0.61      2204
       music        0.67      0.46      0.55       731
     musical        0.44      0.01      0.03       276
     mystery        0.33

model building using NAAIVE BAYES

In [11]:
from sklearn.naive_bayes import MultinomialNB
nb_model=MultinomialNB()
nb_model.fit(x_train_tfidf,y_train)

MultinomialNB()

In [12]:
y_pred_nb = nb_model.predict(x_test_tfidf)
predicted_genres_nb = label_encoder.inverse_transform(y_pred_nb)
test_df['Predicted_Genre_NB'] = predicted_genres_nb
merged_df_nb = pd.merge(test_solution_df,test_df[['ID', 'Predicted_Genre_NB']],on='ID')
print("Merged DataFrame (Naive Bayes Evaluation):")
print(merged_df_nb.head())

Merged DataFrame (Naive Bayes Evaluation):
   ID                          TITLE          GENRE  \
0  1           Edgar's Lunch (1998)       thriller    
1  2       La guerra de papá (1977)         comedy    
2  3    Off the Beaten Track (2010)    documentary    
3  4         Meu Amigo Hindu (2015)          drama    
4  5              Er nu zhai (1955)          drama    

                                         DESCRIPTION Predicted_Genre_NB  
0   L.R. Brane loves his life - his car, his apar...             drama   
1   Spain, March 1964: Quico is a very naughty ch...             drama   
2   One year in the life of Albin and his family ...       documentary   
3   His father has died, he hasn't spoken with hi...             drama   
4   Before he was known internationally as a mart...             drama   


naaives bayes evaluation

In [13]:
from sklearn.metrics import accuracy_score, classification_report
accuracy_nb = accuracy_score(merged_df_nb['GENRE'], merged_df_nb['Predicted_Genre_NB'])
print(f"Naive Bayes Accuracy: {accuracy_nb:.4f}")
print("\nNaive Bayes Classification Report:")
print(classification_report(merged_df_nb['GENRE'],merged_df_nb['Predicted_Genre_NB'],target_names=label_encoder.classes_))

Naive Bayes Accuracy: 0.5092

Naive Bayes Classification Report:
               precision    recall  f1-score   support

      action        0.56      0.03      0.06      1314
       adult        0.46      0.02      0.04       590
   adventure        0.77      0.04      0.08       775
   animation        0.00      0.00      0.00       498
   biography        0.00      0.00      0.00       264
      comedy        0.53      0.40      0.46      7446
       crime        0.00      0.00      0.00       505
 documentary        0.56      0.89      0.69     13096
       drama        0.44      0.84      0.58     13612
      family        0.00      0.00      0.00       783
     fantasy        0.00      0.00      0.00       322
   game-show        1.00      0.02      0.04       193
     history        0.00      0.00      0.00       243
      horror        0.77      0.23      0.35      2204
       music        0.89      0.02      0.05       731
     musical        0.00      0.00      0.00       276

In [15]:
# Assuming models (lr_model, nb_model, svm_model) are already trained
# and 'vectorizer' + 'label_encoder' are fitted
#USECASE
# Sample test descriptions
zoner_Description = [
    'Explosive fight scenes in the city streets',   # Action
    'Haunted mansion that traps its visitors',      # Horror
    'A brave adventurer in search of lost treasure',# Adventure
    'A forbidden romance in the 1920s',             # Romance
    'A daring rescue mission with a love interest'  # Action
]

# Step 1: Vectorize the new test data
test_data_tfidf = vectorizer.transform(zoner_Description)
# Logistic Regression
y_pred_lr = lr_model.predict(test_data_tfidf)
predicted_genres_lr = label_encoder.inverse_transform(y_pred_lr)

# Naive Bayes
y_pred_nb = nb_model.predict(test_data_tfidf)
predicted_genres_nb = label_encoder.inverse_transform(y_pred_nb)
# Step 3: Display predictions
print("Predicted Genres using Logistic Regression :", predicted_genres_lr)
print("Predicted Genres using Naive Bayes        :", predicted_genres_nb)
print()

# Step 4: Detailed comparison per description
for i, message in enumerate(zoner_Description):
    print(f"Story {i+1}: {message}")
    print(f"\tLogistic Regression Prediction: {predicted_genres_lr[i]}")
    print(f"\tNaive Bayes Prediction        : {predicted_genres_nb[i]}")
    print("="*100)

Predicted Genres using Logistic Regression : [' documentary ' ' horror ' ' adventure ' ' drama ' ' drama ']
Predicted Genres using Naive Bayes        : [' documentary ' ' horror ' ' documentary ' ' drama ' ' drama ']

Story 1: Explosive fight scenes in the city streets
	Logistic Regression Prediction:  documentary 
	Naive Bayes Prediction        :  documentary 
Story 2: Haunted mansion that traps its visitors
	Logistic Regression Prediction:  horror 
	Naive Bayes Prediction        :  horror 
Story 3: A brave adventurer in search of lost treasure
	Logistic Regression Prediction:  adventure 
	Naive Bayes Prediction        :  documentary 
Story 4: A forbidden romance in the 1920s
	Logistic Regression Prediction:  drama 
	Naive Bayes Prediction        :  drama 
Story 5: A daring rescue mission with a love interest
	Logistic Regression Prediction:  drama 
	Naive Bayes Prediction        :  drama 
